In [37]:
import os    
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from tqdm.auto import tqdm
from torchvision.datasets import ImageFolder
from torchvision import transforms 
from PIL import Image
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from transformers import get_cosine_schedule_with_warmup
import itertools

## Sampler

In [38]:
class Sampler:
    def __init__(self, num_steps = 1000, beta_start = 0.0001, beta_end = 0.02):
        self.num_steps = num_steps
        self.beta_start = beta_start
        self.beta_end = beta_end

        self.beta_schedule = self.linear_beta_schedule()
        print("Beta Schedule:")  
        print(self.beta_schedule[:5], "...")   # Das ist im Endeffekt die Liste der Beta-Werte, die wir für die Diffusion verwenden werden.
                                    # Heißt Mean = 0, Variance = Beta-Wert, den wir hier berechnet haben. Je höher der Beta-Wert, desto mehr Rauschen wird hinzugefügt.

        self.alpha = 1 - self.beta_schedule
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=-1)  # Das ist die kumulative Produkt der Alpha-Werte. Es wird verwendet, um die Varianz der Verteilung zu berechnen, die wir für die Diffusion verwenden werden.
                                                                # Sodass wir ned jeden Schritt neu berechnen müssen, sondern einfach die kumulative Produkt verwenden können, um die Varianz zu berechnen.
        print("Alpha Cumulative Product:")
        print(self.alpha_cumprod[:5], "...")
        
    def linear_beta_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.num_steps)
    
    def _repeated_unsqueeze(self, target, tensor):
        while target.dim() < tensor.dim():
            target = target.unsqueeze(1)

        return target
    
    def add_noise(self, image, timesteps):
        
        batch_size, c, h, w = image.shape

        device = image.device

        alpha_cumulative_prod_timesteps = self.alpha_cumprod[timesteps].to(device) # Das ist die kumulative Produkt der Alpha-Werte für die gegebenen Timesteps. Es wird verwendet, um die Varianz der Verteilung zu berechnen, die wir für die Diffusion verwenden werden.

        mean_coeff = alpha_cumulative_prod_timesteps.sqrt() 
        var_coeff = (1 - alpha_cumulative_prod_timesteps).sqrt()

        noise = torch.randn_like(image)  # Gaussian Noise mit Input vom Image

        mean_coeff = self._repeated_unsqueeze(mean_coeff, image)
        var_coeff = self._repeated_unsqueeze(var_coeff, image)

        print(image.shape, mean_coeff.shape, var_coeff.shape, noise.shape)

        noise_image = mean_coeff * image + var_coeff * noise 

        return noise_image, noise

    def remove_noise(self, image, timesteps, predicted_noise): # Das ist NICHT das model, einfach nur wird das Noise das predicted wurde nun abgezogen vom noise_image
        
        b, c, h, w = image.shape
        device = image.device
        equal_to_zero_mask = (timesteps == 0)

        beta_t = self.beta_schedule[timesteps].to(device)
        alpha_t = self.alpha[timesteps].to(device)
        alpha_cumprod_t = self.alpha_cumprod[timesteps].to(device)
        alpha_cumprod_prev_t = self.alpha_cumprod[timesteps - 1].to(device)

        alpha_cumprod_prev_t[equal_to_zero_mask] = 1.0

        noise = torch.randn_like(image)

        variance = beta_t * (1 - alpha_cumprod_prev_t) / (1 - alpha_cumprod_t)

        variance = self._repeated_unsqueeze(variance, image)

        variance = torch.sqrt(variance)

        sigma_t_z = variance * noise

        noise_coefficient = beta_t / torch.sqrt(1 - alpha_cumprod_t)
        noise_coefficient = self._repeated_unsqueeze(noise_coefficient, image)

        reciprocal_root_a_t = alpha_t ** -0.5
        reciprocal_root_a_t = self._repeated_unsqueeze(reciprocal_root_a_t, image)

        mean = reciprocal_root_a_t * (image - (noise_coefficient * predicted_noise))

        denoised = mean + sigma_t_z

        return denoised

    
sampler = Sampler()
betas = sampler.linear_beta_schedule()
rand = torch.randn(4,3,64,64)  #  Es hat die gleiche Form wie die Bilder, die wir verwenden werden (Batchgröße, Kanäle, Höhe, Breite).
predicted_noise = torch.randn(4,3,64,64)  # Das ist das Rauschen, das wir vom Modell vorhergesagt bekommen haben. Es hat die gleiche Form wie die Bilder, die wir verwenden werden (Batchgröße, Kanäle, Höhe, Breite).
randtime = torch.randint(0, 1000, (4,))

#sampler.add_noise(rand, randtime);
sampler.remove_noise(rand, randtime, predicted_noise)

Beta Schedule:
tensor([1.0000e-04, 1.1992e-04, 1.3984e-04, 1.5976e-04, 1.7968e-04]) ...
Alpha Cumulative Product:
tensor([0.9999, 0.9998, 0.9996, 0.9995, 0.9993]) ...


tensor([[[[ 2.7162e-01,  1.2976e+00,  1.6012e+00,  ...,  7.2789e-01,
           -1.6878e+00, -1.0567e+00],
          [-2.1379e-01, -1.9928e+00, -1.8222e-01,  ..., -1.2857e+00,
            1.7696e-01, -1.2888e+00],
          [ 1.7755e+00,  8.2133e-01,  5.5119e-01,  ...,  3.7067e-02,
           -7.2347e-01,  2.8737e-01],
          ...,
          [ 1.2304e+00,  2.4284e-01,  5.0584e-01,  ..., -9.5136e-01,
            7.1434e-01, -1.9415e+00],
          [-4.5721e-01, -5.4476e-01,  3.3802e-02,  ...,  1.0268e+00,
            3.1097e-01,  8.6573e-01],
          [ 1.4983e+00,  2.1161e+00,  1.0330e+00,  ..., -1.0177e+00,
           -1.5563e+00,  1.3594e+00]],

         [[-4.2643e-01,  1.2080e+00, -1.5640e+00,  ..., -3.0964e-01,
           -2.6969e+00, -6.1501e-01],
          [ 2.3692e-01, -1.6589e+00, -1.8631e+00,  ...,  5.9748e-01,
            1.4976e-01, -1.3244e+00],
          [-1.3021e+00, -1.0614e-01, -4.7355e-01,  ...,  2.3826e-01,
           -1.5459e+00, -1.0568e+00],
          ...,
     

## Self Attention

<img src="images/Self_Attention.png" width="20%">

In [39]:
from torch import nn
import torch.nn.functional as F


class SelfAttention(nn.Module):
    def __init__(self, in_channels, num_heads=12, attn_p=0, proj_p=0):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.scale = self.head_dim ** -0.5

        self.query = nn.Linear(in_channels, in_channels)        # Diese Daten werden nach und nach angepasst, starten random und werden dann durch das Training immer besser. 
        self.value = nn.Linear(in_channels, in_channels)        # Es ist wichtig, dass die Dimensionen der Query, Key und Value gleich sind, damit wir die Attention berechnen können.
        self.key = nn.Linear(in_channels, in_channels)          

        self.attn_p = attn_p
        self.proj_p = nn.Linear(in_channels, in_channels)
        self.proj_drop = nn.Dropout(proj_p)
    
    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        q = self.query(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_p)
        
        x = x.transpose(1,2).reshape(batch_size, seq_len, embed_dim)
        x = self.proj_drop(self.proj_p(x))
        return x


In [40]:
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) Block.
    
    Warum ist dieses MLP so wichtig?
    Während die Self-Attention-Schicht dafür zuständig ist, Informationen 
    ZWISCHEN verschiedenen Pixeln oder Sequenzpositionen auszutauschen 
    (räumlicher Kontext), arbeitet das MLP isoliert auf jedem einzelnen Pixel. 
    
    Es nimmt die durch die Attention neu gesammelten Informationen (die Features/Kanäle) 
    eines Pixels und verarbeitet diese tiefgreifend in sich selbst. Das MLP mischt 
    also nur entlang der Feature-Dimension. 
    
    Durch die Expansion der Kanäle (meist das 2- bis 4-fache in der Mitte) und 
    die nicht-lineare Aktivierungsfunktion (GELU) erhält das Modell hier den 
    nötigen "Denkraum", um komplexe Muster zu speichern und die gesammelten 
    Erkenntnisse der Attention zu festigen.
    
    Zusammenfassung der Aufgabenteilung im Transformer:
    - Attention = Informationsaustausch über das gesamte Bild (Kommunikation).
    - MLP = Tiefe Verarbeitung der gesammelten Informationen pro Pixel (Einzelarbeit).
    """
    def __init__(self, in_channels, mlp_ratio=4, mlp_p=0):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels * mlp_ratio)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(mlp_p)
        self.fc2 = nn.Linear(in_channels * mlp_ratio, in_channels)
        self.drop2 = nn.Dropout(mlp_p)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x= self.fc2(x)
        x = self.drop2(x)
        return x
    


In [41]:
class TransformerBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 num_heads=4, 
                 mlp_ratio=2,
                 proj_p=0,
                 attn_p=0,
                 mlp_p=0):
        
        super().__init__()
        self.norm1 = nn.LayerNorm(in_channels, eps=1e-6)

        self.attn = SelfAttention(in_channels=in_channels,
                                  num_heads=num_heads, 
                                  attn_p=attn_p,
                                  proj_p=proj_p)
        
        self.norm2 = nn.LayerNorm(in_channels, eps=1e-6)
        self.mlp = MLP(in_channels=in_channels,
                       mlp_ratio=mlp_ratio,
                       mlp_p=mlp_p)
        
    def forward(self, x):
        batch_size, channels, height, width = x.shape
      
        ### Reshape to batch_size x (height*width) x channels
        x = x.reshape(batch_size, channels, height*width).permute(0,2,1)
        
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))

        x = x.permute(0,2,1).reshape(batch_size, channels, height, width)
        return x

In [42]:
rand_img = torch.randn(4,32,64,64)
t = TransformerBlock(in_channels=32)
t(rand_img).shape

torch.Size([4, 32, 64, 64])

## Sinusoidal Time Embeddings

In [43]:
class SinusoidalTimeEmbeddings(nn.Module):
    def __init__(self, time_embed_dim, scaled_time_embed_dim):
        super().__init__()

        self.inv_freq = nn.Parameter(1.0/(10000 ** (torch.arange(0, time_embed_dim, 2).float() / time_embed_dim)), requires_grad=False)

        self.time_mlp = nn.Sequential(nn.Linear(time_embed_dim, scaled_time_embed_dim), 
                                      nn.SiLU(), 
                                      nn.Linear(scaled_time_embed_dim, scaled_time_embed_dim),
                                      nn.SiLU())
    
    def forward(self, timesteps):
        time_step_freqs = timesteps.unsqueeze(1) * self.inv_freq.unsqueeze(0)
        embeddings = torch.cat([torch.sin(time_step_freqs), torch.cos(time_step_freqs)], dim=-1)
        embeddings = self.time_mlp(embeddings)
        return embeddings

s = SinusoidalTimeEmbeddings(128, 256)
timesteps = torch.tensor([1,2,3])
s(timesteps).shape

torch.Size([3, 256])

## U-Net

<img src="images/U-Net.png" width="70%">

In [44]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, groupnorm_num_groups, time_embed_dim):
        super().__init__()

        self.time_expand = nn.Linear(time_embed_dim, out_channels)
        self.groupnorml = nn.GroupNorm(groupnorm_num_groups, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding="same")

        self.groupnorm2 = nn.GroupNorm(groupnorm_num_groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding="same")

        self.resize_channels = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()


    def forward(self, x, time_embeddings):
        residual_connection = x                             # Speicherung des Alten wertes x
        time_embed = self.time_expand(time_embeddings)

        x = self.groupnorml(x)
        x = F.silu(x)
        x = self.conv1(x)

        x = x + time_embed.unsqueeze(-1).unsqueeze(-1)

        x = self.groupnorm2(x)
        x = F.silu(x)
        x = self.conv2(x)                                   # Berechnung von F(x)

        x = x + self.resize_channels(residual_connection)   # y = F(x) + x

        return x

    

rand = torch.randn(4,64 ,128, 128)
time_embeds = torch.randn(4,256)
rb = ResidualBlock(64, 64, 16, 256)

rb(rand, time_embeds);

In [45]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding="same")
        )

    def forward(self, x):
        return self.upsample(x)

<img src="images/U-Net_2.png" width="70%">

In [46]:

class UNET(nn.Module):
    def __init__(self, in_channels=3, start_dim = 64, dim_mults=(1,2,4), residual_blocks_per_group=1, groupnorm_num_groups=16, time_embed_dim=128):
        super().__init__()

        self.input_image_channels = in_channels

        channel_sizes = [start_dim * i for i in dim_mults]
        starting_channel_size, ending_channel_size = channel_sizes[0], channel_sizes[-1]

        self.encoder_config = []

        for idx, d in enumerate(channel_sizes):
            for _ in range(residual_blocks_per_group):
                self.encoder_config.append(((d, d), "residual"))

            self.encoder_config.append(((d,d), "downsample"))

            self.encoder_config.append((d, "attention"))

            if idx < len(channel_sizes) - 1:
                self.encoder_config.append(((d, channel_sizes[idx+1]), "residual"))

        self.bottleneck_config = []
        for _ in range(residual_blocks_per_group):
            self.bottleneck_config.append(((ending_channel_size, ending_channel_size), "residual"))

        out_dim = ending_channel_size
        reversered_encoder_config = self.encoder_config[::-1]

        self.decoder_config = []
        for idx, (metadata, l_type) in enumerate(reversered_encoder_config):
            
            if l_type != "attention":
                enc_in_channels, enc_out_channels = metadata
                self.decoder_config.append(
                    (
                        (out_dim+enc_out_channels, enc_in_channels), "residual"
                    )
                )

                if l_type == "downsample":
                    self.decoder_config.append(
                        (
                            (enc_in_channels, enc_in_channels), "upsample"
                        )
                    )
                out_dim = enc_in_channels

            else:
                in_channels = metadata
                self.decoder_config.append(
                    (
                        in_channels, "attention"
                    )
                )
        self.decoder_config.append(((starting_channel_size * 2, starting_channel_size), "residual"))

        ### ACTUALLY BUILD MODEL ###

        self.conv_in_proj = nn.Conv2d(self.input_image_channels, starting_channel_size, kernel_size=3, padding="same")

        self.encoder = nn.ModuleList()

        for metadata, l_type in self.encoder_config:
            if l_type == "residual":
                in_channels, out_channels = metadata
                self.encoder.append(ResidualBlock(in_channels, out_channels, groupnorm_num_groups, time_embed_dim))
            elif l_type == "downsample":
                in_channels, out_channels = metadata
                self.encoder.append(
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
                )
            elif l_type == "attention":
                in_channels = metadata
                self.encoder.append(TransformerBlock(in_channels))
        print(self.decoder_config)

        self.bottleneck = nn.ModuleList()
        for (in_channels, out_channels), _ in self.bottleneck_config:
            self.bottleneck.append(ResidualBlock(in_channels, out_channels, groupnorm_num_groups, time_embed_dim))

        self.decoder = nn.ModuleList()
        for metadata, l_type in self.decoder_config:
            if l_type == "residual":
                in_channels, out_channels = metadata
                self.decoder.append(ResidualBlock(in_channels, out_channels, groupnorm_num_groups, time_embed_dim))
            elif l_type == "upsample":
                in_channels, out_channels = metadata
                self.decoder.append(UpsampleBlock(in_channels, out_channels))
            elif l_type == "attention":
                in_channels = metadata
                self.decoder.append(TransformerBlock(in_channels))

        self.conv_out_proj = nn.Conv2d(starting_channel_size, self.input_image_channels, kernel_size=3, padding="same")

    def forward(self, x, time_embeddings):
        residuals = []

        x = self.conv_in_proj(x)
        residuals.append(x)  # wieder rein!

        for module in self.encoder:
            if isinstance(module, ResidualBlock):
                x = module(x, time_embeddings)
                residuals.append(x)
            elif isinstance(module, nn.Conv2d):  # Downsample
                x = module(x)
                residuals.append(x)
            else:  # Attention
                x = module(x)

        for module in self.bottleneck:
            x = module(x, time_embeddings)

        for module in self.decoder:
            if isinstance(module, ResidualBlock):
                residual_tensor = residuals.pop()
                #print(residual_tensor.shape, x.shape)
                x = torch.cat([x, residual_tensor], dim=1)
                #print(x.shape)
                x = module(x, time_embeddings)
            else:
                x = module(x)

        x = self.conv_out_proj(x)
        return x
                 


        
#
#unet = UNET()
#rand = torch.randn(4,3,128,128)
#time_embeds = torch.randn(4,128)
#unet(rand, time_embeds)
#

In [47]:
class Diffusion(nn.Module):
    def __init__(self, 
                 in_channels=3, 
                 start_dim=64, 
                 dim_mults=(1,2,4,4), 
                 residual_blocks_per_group=1, 
                 groupnorm_num_groups=16, 
                 time_embed_dim=128, 
                 time_embed_dim_ratio=2):
        super().__init__()
        self.in_channels = in_channels
        self.start_dim = start_dim
        self.dim_mults = dim_mults
        self.residual_blocks_per_group = residual_blocks_per_group
        self.groupnorm_num_groups = groupnorm_num_groups

        self.time_embed_dim = time_embed_dim
        self.scaled_time_embed_dim = int(time_embed_dim * time_embed_dim_ratio)

        self.sinusoidal_time_embeddings = SinusoidalTimeEmbeddings(time_embed_dim, self.scaled_time_embed_dim)

        self.unet = UNET(in_channels=in_channels, 
                         start_dim=start_dim, 
                         dim_mults=dim_mults, 
                         residual_blocks_per_group=residual_blocks_per_group, 
                         groupnorm_num_groups=groupnorm_num_groups, 
                         time_embed_dim=self.scaled_time_embed_dim)
        
        def forward(self, noisy_images, timesteps):
            timesecond_embeddings = self.sinusoidal_time_embeddings(timesteps)
            predicted_noise = self.unet(noisy_images, timesecond_embeddings)
            return predicted_noise



### Helper Function

In [48]:
@torch.no_grad()
def sample_plot_image(step_idx, 
                      total_timesteps, 
                      sampler, 
                      image_size,
                      num_channels,
                      plot_freq, 
                      model,
                      num_gens,
                      path_to_generated_dir,
                      device):

    tensor2image_transform = transforms.Compose([
        transforms.Lambda(lambda t: t.squeeze(0)),
        transforms.Lambda(lambda t: (t + 1) / 2),
        transforms.Lambda(lambda t: t.permute(1, 2, 0)),
        transforms.Lambda(lambda t: t * 255.),
        transforms.Lambda(lambda t: t.cpu().numpy().astype(np.uint8)),
        transforms.ToPILImage(),
    ])

    images = torch.randn((num_gens, num_channels, image_size, image_size))
    num_images_per_gen = (total_timesteps // plot_freq) + 1

    images_to_vis = [[] for _ in range(num_gens)]

    for t in tqdm(range(total_timesteps, -1, -1)):
        ts = torch.full((num_gens,) , t)
        noise_pred = model(images.to(device), ts.to(device))
        images = sampler.remove_noise(images, ts, noise_pred)
        if t % plot_freq == 0:
            for gen_idx in range(num_gens):
                images_to_vis[gen_idx].append(tensor2image_transform(images[gen_idx]))

    images_to_vis = list(itertools.chain(*images_to_vis))

    fig, axes = plt.subplots(nrows=num_gens, ncols=num_images_per_gen, figsize=(num_images_per_gen, num_gens))
    plt.tight_layout()

    for ax, image in zip(axes.ravel(), images_to_vis):
        ax.imshow(image)
        ax.axis("off")
    fig.subplots_adjust(wspace=0.05, hspace=0.05)
    plt.savefig(os.path.join(path_to_generated_dir, f"gen_{step_idx}.png"))
    plt.show()
    plt.close()


In [51]:
def train(image_size=64,
          evaluation_interval=5000,
          total_timesteps=500,
          plot_freq_interval=50,
          num_generations=5,
          num_training_steps=50000,
          num_input_channels=3,
          batch_size=64,
          path_to_generated="work_dir"):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    image2tensor = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Lambda(lambda t: (t * 2) - 1)
    ])

    dataset = ImageFolder("data", transform=image2tensor)
    trainloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)

train()



FileNotFoundError: [WinError 3] Das System kann den angegebenen Pfad nicht finden: 'data'